# EdNet: CDLD and IRT on the same response-prediction tasks

A reviewer’s reproducibility notebook / 리뷰어 재현용 노트북

## Research question
Does cyclic discovery of multidimensional user and item latents improve response prediction over Rasch and 2PL? The two notebooks share all model, splitting, evaluation and analysis code. Only raw-data conversion and dataset paths differ.

학생·문항의 다차원 잠재값 발견이 Rasch·2PL보다 응답 예측을 개선하는지 비교합니다. 전처리와 자료 경로를 제외한 코드는 두 노트북에서 같습니다.

## How to read and run
1. Read the protocol below, then set the input paths. Use Python 3.11 and the pinned environment.
2. For a first check, choose `RUN_MODE="smoke"`; its outputs are explicitly excluded from paper results.
3. For the full experiment, choose `RUN_MODE="full"` and Run All. Progress is printed during long fits.
4. To inspect saved results, restart the kernel and run Environment, Data paths, Configuration, then Saved-results analysis and Paper figures. No model training is needed.

처음에는 경로를 맞추고 smoke로 실행 경로를 점검한 뒤 full로 실행합니다. 결과·그림은 재학습 없이 재생성할 수 있습니다.

## What is being reproduced
This is a new harmonized, response-only comparison, not a byte-for-byte reproduction of the earlier EdNet paper or the earlier 180-item KMLE evaluation. Both datasets now use 500 selected students, 40 disjoint query responses per student, and nested adaptation sets up to 40 responses. Selection requires at least 80 distinct responses to estimable items. Existing students and entirely excluded new students are evaluated on the same queries. This is random held-out-response prediction, not chronological forecasting, unseen-item prediction, or validation of licensing pass/fail decisions.

기존 연구의 수치를 복제하는 노트북이 아니라, 두 자료의 조건을 맞춘 새 비교 실험입니다. 학생당 평가40문항과 적응 최대40문항을 분리하며, 사용 가능한 응답이 충분한 학생 집단에 대한 결과입니다. 시간 순 예측이나 신규 문항 예측으로 해석하지 않습니다.

## Interpretation agreed before this run
The primary outputs are response AUC and observed query-score correlation. Rasch/2PL are comparators, not known ceilings. The historical correlation margin 0.010 is a stated practical criterion, not an independently validated educational threshold; it is not an AUC margin. No overall “IRT sufficiency for test construction” or “CDLD non-inferiority” is inferred from these two metrics alone. The hypotheses were developed after earlier KMLE results and are not an independent preregistered confirmation.

정답 예측 성능만으로 시험 구성의 충분성이나 능력 측정의 타당성을 선언하지 않습니다. 기존 KMLE 결과를 본 뒤 구성한 후속 비교이며, 비열등성 기준의 범위와 근거를 제한적으로 명시합니다.

## Data source
[Official EdNet repository](https://github.com/riiid/ednet): download KT1 and Contents; only `questions.csv` is needed from Contents. EdNet is distributed under CC BY-NC 4.0. Keep the last response per person–item pair, using stable timestamp order. This follows the earlier EdNet preprocessing choice; it is not a model of temporal learning. Shared question bundles can create within-person dependence; this is not a held-out-bundle test. Question tags, parts and elapsed times are not model inputs.


Common code SHA-256: `88e6286bc435766557e9730555dced35f7b175a6f4702e84c41bd28748be1e12`

The other dataset notebook must display the same hash. / 다른 자료의 노트북에도 같은 해시가 표시되어야 합니다.

In [ ]:
COMMON_CODE_SHA256 = "88e6286bc435766557e9730555dced35f7b175a6f4702e84c41bd28748be1e12"


## Optional installation / 선택 설치
Use a fresh Python 3.11 kernel. Linux installation includes the CUDA libraries; use the existing NVIDIA driver supplied by the instance.
Python 3.11 커널을 사용합니다. Vast.ai에서 기존 드라이버를 유지하고 필요한 Python 패키지만 설치합니다.

In [ ]:
# Optional: run once in a clean kernel, then restart it before continuing.
INSTALL_DEPENDENCIES = False
import sys
import subprocess
import platform
required_packages = [
    "tensorflow[and-cuda]==2.21.0" if platform.system() == "Linux" else "tensorflow==2.21.0",
    "keras==3.15.1", "numpy==2.4.6", "scipy==1.17.1", "scikit-learn==1.9.1",
    "pandas==3.0.5", "matplotlib==3.11.2",
]
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *required_packages])
print("Install only if needed; use the pinned versions for the reproducibility run.")


# 1. Environment and data

## Environment
Import the numerical and plotting tools. GPU memory grows as needed when TensorFlow is loaded later.

수치계산·그림 도구를 준비합니다. 뒤에서 TensorFlow를 불러올 때 GPU 메모리를 필요한 만큼 사용하도록 설정합니다.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "4")
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "1")
import gc
import sys
import json
import time
import hashlib
import zipfile
import traceback
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.special import expit, logsumexp
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss, cohen_kappa_score
from IPython.display import display

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "pdf.fonttype": 42, "savefig.dpi": 300})
print("Python", sys.version.split()[0])


## Data paths / 데이터 경로
Edit only these paths to match your instance. Do not put data from the two datasets in the same results directory.
실제 파일 위치에 맞게 아래 경로를 지정합니다. 두 자료의 결과 디렉터리는 자동으로 분리됩니다.

In [ ]:
DATASET_NAME = "EdNet"
WORK_DIR = Path.cwd()
NOTEBOOK_PATH = WORK_DIR / "02_ednet_cdld_irt_01.ipynb"
RAW_DATA = WORK_DIR / "data" / "EdNet-KT1.zip"
QUESTIONS_FILE = WORK_DIR / "data" / "questions.csv"


## One shared experimental configuration
These scientific settings are identical in both notebooks. Full mode is the reported experiment; smoke mode only checks that the pipeline runs.

두 노트북의 실험 설정은 같습니다. full은 본실험이고 smoke는 실행 경로를 확인하는 소규모 점검입니다.

In [ ]:
# Identical scientific settings in the KMLE and EdNet notebooks.
RUN_MODE = "full"                 # "smoke" is a pipeline check, never a paper result.
RUN_EXPERIMENT = True             # False: read saved results without training.
RESUME = True
CONFIG = {
    "seeds": [42, 43, 44, 45, 46],
    "protocols": ["warm", "cold"],
    "evaluation_students": 500,
    "support_counts": [0, 4, 8, 12, 16, 20, 40],
    "query_count": 40,
    "validation_fraction": 0.1,
    "latent_size": 64,
    "batch_size": 8192,
    "finder_cycles": 5,
    "finder_subepochs": 10,
    "predictor_epochs": 25,
    "adaptation_epochs": 300,
    "learning_rate": 0.0005,
    "irt_nodes": 41,
    "irt_max_iterations": 3000,
    "irt_tolerance": 1e-5,
    "irt_person_batch": 4096,
    "bootstrap_draws": 2000,
    "bootstrap_seed": 20260914,
    "correlation_margin": 0.010,
}
if RUN_MODE == "smoke":
    CONFIG.update(seeds=[42], evaluation_students=8, support_counts=[0, 4, 8],
                  query_count=8, finder_cycles=1, finder_subepochs=1,
                  predictor_epochs=1, adaptation_epochs=2, bootstrap_draws=100)
else:
    assert RUN_MODE == "full"
OUTPUT_DIR = WORK_DIR / "results" / DATASET_NAME / RUN_MODE
CACHE_DIR = WORK_DIR / "cache" / DATASET_NAME / RUN_MODE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
display(pd.DataFrame({"setting": CONFIG.keys(), "value": [str(v) for v in CONFIG.values()]}))
print("Results:", OUTPUT_DIR)


## Verified data caches and atomic saves
A cache is reused only when its source and content checksums agree. Temporary files prevent an interrupted write from appearing complete.

원자료와 내용의 해시가 일치하는 캐시만 재사용합니다. 저장 도중 중단된 파일은 완료본으로 인식하지 않습니다.

In [ ]:
RESPONSE_DTYPE = np.dtype([("person", "<i4"), ("item", "<i4"),
                           ("correct", "u1"), ("timestamp", "<i8")])

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def save_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=True))
    temporary.replace(path)

def save_npz(path, **values):
    path = Path(path)
    temporary = path.with_name(path.stem + ".tmp.npz")
    np.savez_compressed(temporary, **values)
    temporary.replace(path)

def finish_cache(parts, people, items, description):
    counts = [len(np.load(path, mmap_mode="r")) for path in parts]
    target = CACHE_DIR / "responses.npy"
    temporary = CACHE_DIR / "responses.pending.npy"
    responses = np.lib.format.open_memmap(temporary, mode="w+", dtype=RESPONSE_DTYPE,
                                         shape=(sum(counts),))
    start = 0
    for path, count in zip(parts, counts):
        responses[start:start + count] = np.load(path, mmap_mode="r")
        start += count
    responses.flush()
    del responses
    temporary.replace(target)
    people.to_csv(CACHE_DIR / "people.csv", index=False)
    items.to_csv(CACHE_DIR / "items.csv", index=False)
    description.update(rows=sum(counts), people=len(people), items=len(items),
                       response_sha256=file_sha256(target))
    save_json(CACHE_DIR / "complete.json", description)
    return np.load(target, mmap_mode="r"), description

def load_cache(expected_sources):
    marker = CACHE_DIR / "complete.json"
    if not marker.exists():
        return None
    description = json.loads(marker.read_text())
    if description["source_hashes"] != expected_sources:
        raise RuntimeError("The input changed. Use a new cache directory; do not mix datasets.")
    if file_sha256(CACHE_DIR / "responses.npy") != description["response_sha256"]:
        raise RuntimeError("The response cache checksum failed.")
    print("Verified cached responses:", description["rows"])
    return np.load(CACHE_DIR / "responses.npy", mmap_mode="r"), description


## Dataset-specific preprocessing / 자료별 전처리
Read KT1 student files in bounded chunks, score from the answer key, and cache unique observed pairs.

이 셀만 자료 형식에 따라 다릅니다. 이후 공통 코드는 person·item·correct만 사용합니다.

In [ ]:
def prepare_data():
    if not RAW_DATA.is_file() or not QUESTIONS_FILE.is_file():
        raise FileNotFoundError("Provide EdNet-KT1.zip and the Contents questions.csv in the paths above.")
    source_hashes = {RAW_DATA.name: file_sha256(RAW_DATA), QUESTIONS_FILE.name: file_sha256(QUESTIONS_FILE)}
    cached = load_cache(source_hashes)
    if cached is not None:
        return cached
    questions = pd.read_csv(QUESTIONS_FILE)
    if questions["question_id"].duplicated().any():
        raise ValueError("Duplicate question IDs in the answer key.")
    questions = questions.sort_values("question_id", key=lambda values: values.str[1:].astype(int)).reset_index(drop=True)
    item_lookup = dict(zip(questions["question_id"], questions.index))
    answer_lookup = dict(zip(questions["question_id"], questions["correct_answer"]))
    items = questions.rename(columns={"question_id": "original_id"}).copy()
    items.insert(0, "item", np.arange(len(items)))
    # Metadata below is saved for audit and later analyses; the models receive none of it.
    parts, rows_read, rows_kept, invalid_answers = [], 0, 0, 0
    with zipfile.ZipFile(RAW_DATA) as archive:
        members = sorted([name for name in archive.namelist() if Path(name).name.startswith("u") and name.endswith(".csv")],
                         key=lambda name: int(Path(name).stem[1:]))
        if RUN_MODE == "smoke":
            members = members[:300]
        people = pd.DataFrame({"person": np.arange(len(members)), "original_id": [Path(name).stem for name in members]})
        for chunk_start in range(0, len(members), 2000):
            part = CACHE_DIR / f"part_{chunk_start // 2000:05d}.npy"
            marker = part.with_suffix(".json")
            if part.exists() and marker.exists():
                info = json.loads(marker.read_text())
                if info["source_hashes"] != source_hashes or file_sha256(part) != info["sha256"]:
                    raise RuntimeError("A preprocessing checkpoint does not match its input.")
            else:
                blocks, raw_count, invalid_count = [], 0, 0
                for person in range(chunk_start, min(chunk_start + 2000, len(members))):
                    with archive.open(members[person]) as stream:
                        frame = pd.read_csv(stream, usecols=["timestamp", "question_id", "user_answer"])
                    raw_count += len(frame)
                    if not frame["question_id"].isin(item_lookup).all():
                        raise ValueError(f"An unknown question ID occurs in {members[person]}")
                    # Stable order resolves equal timestamps by the last row in the source file.
                    frame = frame.sort_values("timestamp", kind="stable").drop_duplicates("question_id", keep="last")
                    valid = frame["user_answer"].isin(["a", "b", "c", "d"])
                    invalid_count += int((~valid).sum())
                    frame = frame.loc[valid].copy()
                    frame["item"] = frame["question_id"].map(item_lookup)
                    frame = frame.sort_values("item")
                    block = np.empty(len(frame), dtype=RESPONSE_DTYPE)
                    block["person"] = person
                    block["item"] = frame["item"].to_numpy()
                    block["correct"] = (frame["user_answer"] == frame["question_id"].map(answer_lookup)).to_numpy()
                    block["timestamp"] = frame["timestamp"].to_numpy()
                    blocks.append(block)
                block = np.concatenate(blocks)
                temporary = part.with_name(part.stem + ".pending.npy")
                np.save(temporary, block)
                temporary.replace(part)
                info = dict(source_hashes=source_hashes, rows_read=raw_count, rows_kept=len(block),
                            invalid_answers=invalid_count, sha256=file_sha256(part))
                save_json(marker, info)
            rows_read += info["rows_read"]
            rows_kept += info["rows_kept"]
            invalid_answers += info["invalid_answers"]
            parts.append(part)
            print(f"Students {min(chunk_start + 2000, len(members)):,}/{len(members):,}; retained {rows_kept:,} responses", flush=True)
    return finish_cache(parts, people, items, {
        "dataset": DATASET_NAME, "source_hashes": source_hashes,
        "deduplication": "Last response per person–item pair (stable timestamp order).",
        "timestamp_available": True, "original_rows": rows_read, "invalid_answers_excluded": invalid_answers,
    })

responses, data_description = prepare_data()
display(pd.DataFrame([data_description]).T.rename(columns={0: "value"}))


# 2. Matched evaluation design

## The same students and query responses in both protocols
Warm evaluation keeps each selected student’s other training responses; cold evaluation excludes that student entirely. Query responses are never used for fitting or early stopping.

기존 학생 평가는 평가 응답을 제외한 학습 응답을 사용하고, 신규 학생 평가는 해당 학생 전체를 학습에서 제외합니다. 평가 응답은 학습·조기종료에 사용하지 않습니다.

In [ ]:
def make_split(data, description, config, seed):
    n_people, n_items = description["people"], description["items"]
    people, items = data["person"], data["item"]
    counts = np.bincount(people, minlength=n_people)
    offsets = np.concatenate(([0], np.cumsum(counts)))
    required = max(config["support_counts"]) + config["query_count"]
    eligible = np.flatnonzero(counts >= required)
    if len(eligible) < config["evaluation_students"]:
        raise ValueError(f"Need {config['evaluation_students']} people with ≥{required} distinct responses; found {len(eligible)}.")
    rng = np.random.default_rng(seed)
    evaluation_people = np.sort(rng.choice(eligible, config["evaluation_students"], replace=False))
    is_evaluation_person = np.zeros(n_people, dtype=bool)
    is_evaluation_person[evaluation_people] = True
    # Split fitting/validation rows once. Nothing is selected using test performance.
    validation = rng.random(len(data), dtype=np.float32) < config["validation_fraction"]
    cold_available = ~is_evaluation_person[people]
    cold_fit = cold_available & ~validation
    successes = np.bincount(items[cold_fit], weights=data["correct"][cold_fit], minlength=n_items)
    trials = np.bincount(items[cold_fit], minlength=n_items)
    usable_item = (successes > 0) & (successes < trials)
    # Constant items have no finite unpenalized difficulty estimate. This train-only
    # exclusion applies to every model and to both evaluation protocols.
    usable_rows = usable_item[items]
    support, query = [], []
    for person in evaluation_people:
        rows = np.arange(offsets[person], offsets[person + 1])
        rows = rows[usable_rows[rows]]
        if len(rows) < required:
            raise ValueError(f"Person {person} has only {len(rows)} responses to estimable items; need {required}. No automatic cohort change.")
        chosen = rng.choice(rows, required, replace=False)
        support.append(chosen[:max(config["support_counts"])])
        query.append(chosen[max(config["support_counts"]):])
    support, query = np.array(support), np.array(query)
    warm_available = usable_rows.copy()
    warm_available[query.ravel()] = False
    cold_available &= usable_rows
    plans = {}
    for protocol, available in [("warm", warm_available), ("cold", cold_available)]:
        train = np.flatnonzero(available & ~validation)
        valid = np.flatnonzero(available & validation)
        assert not np.isin(query.ravel(), train).any()
        assert not np.isin(query.ravel(), valid).any()
        if protocol == "cold":
            assert not is_evaluation_person[people[train]].any()
            assert not is_evaluation_person[people[valid]].any()
        assert not np.intersect1d(support.ravel(), query.ravel()).size
        plans[protocol] = {"train": train, "valid": valid}
    return {"people": evaluation_people, "support": support, "query": query,
            "plans": plans, "usable_item": usable_item,
            "audit": {"eligible_people_before_item_filter": len(eligible),
                      "evaluation_people": len(evaluation_people),
                      "query_per_person": config["query_count"],
                      "items_with_no_training_response": int((trials == 0).sum()),
                      "items_with_only_one_training_outcome": int(((trials > 0) & ~usable_item).sum()),
                      "excluded_rows": int((~usable_rows).sum()),
                      "total_rows": len(data)}}


# 3. Item response theory

## Ability scale and integration points
Rasch fixes discrimination to one; 2PL estimates it. Missing responses contribute no likelihood. The prior, quadrature, parameter bounds and stopping rule are shared.

Rasch는 변별도를 1로 고정하고 2PL은 추정합니다. 미관측 응답을 오답으로 처리하지 않으며 두 자료에 같은 사전분포·수치적분·경계·수렴 기준을 사용합니다.

In [ ]:
def irt_grid(nodes):
    theta = np.linspace(-4.0, 4.0, nodes)
    prior = np.exp(-0.5 * theta ** 2)
    return theta, prior / prior.sum()


## Represent only observed responses
Rasch fixes discrimination to one; 2PL estimates it. Missing responses contribute no likelihood. The prior, quadrature, parameter bounds and stopping rule are shared.

Rasch는 변별도를 1로 고정하고 2PL은 추정합니다. 미관측 응답을 오답으로 처리하지 않으며 두 자료에 같은 사전분포·수치적분·경계·수렴 기준을 사용합니다.

In [ ]:
def sparse_responses(data, rows, n_people, n_items):
    people = data["person"][rows]
    items = data["item"][rows]
    observed = sparse.csr_matrix((np.ones(len(rows)), (people, items)), shape=(n_people, n_items))
    correct = sparse.csr_matrix((data["correct"][rows].astype(float), (people, items)), shape=(n_people, n_items))
    if observed.data.max() != 1:
        raise ValueError("Repeated person–item pairs must be resolved before fitting.")
    return observed, correct


## Posterior weights without imputing missing responses
Rasch fixes discrimination to one; 2PL estimates it. Missing responses contribute no likelihood. The prior, quadrature, parameter bounds and stopping rule are shared.

Rasch는 변별도를 1로 고정하고 2PL은 추정합니다. 미관측 응답을 오답으로 처리하지 않으며 두 자료에 같은 사전분포·수치적분·경계·수렴 기준을 사용합니다.

In [ ]:
def irt_expectation(observed, correct, a, b, theta, prior, person_batch):
    probability = np.clip(expit(a[:, None] * (theta[None, :] - b[:, None])), 1e-9, 1 - 1e-9)
    log_odds = np.log(probability) - np.log1p(-probability)
    log_incorrect = np.log1p(-probability)
    expected_trials = np.zeros_like(probability)
    expected_correct = np.zeros_like(probability)
    likelihood, last_message = 0.0, time.monotonic()
    for start in range(0, observed.shape[0], person_batch):
        obs = observed[start:start + person_batch]
        yes = correct[start:start + person_batch]
        log_posterior = yes @ log_odds + obs @ log_incorrect + np.log(prior)[None, :]
        normalizer = logsumexp(log_posterior, axis=1)
        posterior = np.exp(log_posterior - normalizer[:, None])
        likelihood += float(normalizer.sum())
        expected_trials += obs.T @ posterior
        expected_correct += yes.T @ posterior
        if time.monotonic() - last_message > 30:
            print(f"  IRT expectation: {min(start + person_batch, observed.shape[0]):,}/{observed.shape[0]:,} people", flush=True)
            last_message = time.monotonic()
    return expected_trials, expected_correct, likelihood


## Estimate item parameters and check convergence
Rasch fixes discrimination to one; 2PL estimates it. Missing responses contribute no likelihood. The prior, quadrature, parameter bounds and stopping rule are shared.

Rasch는 변별도를 1로 고정하고 2PL은 추정합니다. 미관측 응답을 오답으로 처리하지 않으며 두 자료에 같은 사전분포·수치적분·경계·수렴 기준을 사용합니다.

In [ ]:
def fit_irt(data, rows, description, config, model, checkpoint):
    observed, correct = sparse_responses(data, rows, description["people"], description["items"])
    theta, prior = irt_grid(config["irt_nodes"])
    trials = np.asarray(observed.sum(axis=0)).ravel()
    successes = np.asarray(correct.sum(axis=0)).ravel()
    active = trials > 0
    initial_rate = np.clip(np.divide(successes, trials, out=np.full_like(trials, 0.5), where=active), .01, .99)
    a, b = np.ones(len(trials)), -np.log(initial_rate / (1 - initial_rate))
    first_iteration, history = 0, []
    checkpoint = Path(checkpoint)
    if RESUME and checkpoint.exists():
        saved = np.load(checkpoint)
        a, b = saved["a"], saved["b"]
        first_iteration = int(saved["iteration"])
        history = saved["history"].tolist()
        print(f"Resuming {model} at iteration {first_iteration}")
    converged = bool(history and history[-1][1] < config["irt_tolerance"])
    iteration = first_iteration - 1
    for iteration in range(first_iteration, config["irt_max_iterations"]):
        if converged:
            iteration = first_iteration - 1
            break
        expected_trials, expected_correct, likelihood = irt_expectation(
            observed, correct, a, b, theta, prior, config["irt_person_batch"])
        old_a, old_b = a.copy(), b.copy()
        for step in range(25):
            probability = np.clip(expit(a[:, None] * (theta[None, :] - b[:, None])), 1e-9, 1 - 1e-9)
            weight = expected_trials * probability * (1 - probability)
            residual = expected_correct - expected_trials * probability
            gradient_b = (-a[:, None] * residual).sum(axis=1)
            hessian_b = -(a[:, None] ** 2 * weight).sum(axis=1)
            b -= gradient_b / np.minimum(hessian_b, -1e-9)
            if model == "2PL":
                probability = np.clip(expit(a[:, None] * (theta[None, :] - b[:, None])), 1e-9, 1 - 1e-9)
                weight = expected_trials * probability * (1 - probability)
                residual = expected_correct - expected_trials * probability
                distance = theta[None, :] - b[:, None]
                gradient_a = (residual * distance).sum(axis=1)
                hessian_a = -(weight * distance ** 2).sum(axis=1)
                a = np.clip(a - gradient_a / np.minimum(hessian_a, -1e-9), .1, 4.0)
        change = float(max(np.max(abs(a - old_a)), np.max(abs(b - old_b))))
        if not np.isfinite(change):
            raise FloatingPointError(f"Nonfinite {model} update")
        history.append([iteration + 1, change, likelihood])
        converged = change < config["irt_tolerance"]
        if iteration == 0 or (iteration + 1) % 10 == 0 or converged:
            print(f"{model}: iteration {iteration+1}, max parameter change={change:.3g}", flush=True)
            save_npz(checkpoint, a=a, b=b, iteration=iteration + 1, history=np.asarray(history))
        if converged:
            break
    del observed, correct
    gc.collect()
    return {"a": a, "b": b, "active": active, "converged": converged,
            "iterations": iteration + 1, "history": np.asarray(history),
            "slope_boundary_items": int((((a <= .1) | (a >= 4.0)) & active).sum())}


## Predict from permitted response history
Rasch fixes discrimination to one; 2PL estimates it. Missing responses contribute no likelihood. The prior, quadrature, parameter bounds and stopping rule are shared.

Rasch는 변별도를 1로 고정하고 2PL은 추정합니다. 미관측 응답을 오답으로 처리하지 않으며 두 자료에 같은 사전분포·수치적분·경계·수렴 기준을 사용합니다.

In [ ]:
def irt_predict(data, support_rows, query_rows, fitted, config):
    theta, prior = irt_grid(config["irt_nodes"])
    item_probability = np.clip(expit(fitted["a"][:, None] * (theta[None, :] - fitted["b"][:, None])), 1e-9, 1 - 1e-9)
    predictions, estimates = [], []
    for support, query in zip(support_rows, query_rows):
        probability = item_probability[data["item"][support]]
        labels = data["correct"][support, None]
        log_posterior = (labels * np.log(probability) + (1 - labels) * np.log1p(-probability)).sum(axis=0) + np.log(prior)
        posterior = np.exp(log_posterior - logsumexp(log_posterior))
        predictions.append(item_probability[data["item"][query]] @ posterior)
        estimates.append(posterior @ theta)
    return np.asarray(predictions), np.asarray(estimates)


# 4. Cyclic dual latent discovery

## CDLD model definitions
Two Finders alternate user/item latent discovery, followed by a separate Predictor. Zero-valued feature slots retain the reference architecture without adding metadata.

두 Finder가 학생·문항 잠재값을 교대로 발견하고 별도 Predictor가 예측합니다. 특성 입력은 0으로 두어 메타데이터 없이 정본 구조를 유지합니다.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, concatenate, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.losses import Loss
from keras import layers, utils

tf.get_logger().setLevel("ERROR")
for physical_gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(physical_gpu, True)
print("TensorFlow", tf.__version__, "| GPUs:", len(tf.config.list_physical_devices("GPU")))


## Look up a response index
Two Finders alternate user/item latent discovery, followed by a separate Predictor. Zero-valued feature slots retain the reference architecture without adding metadata.

두 Finder가 학생·문항 잠재값을 교대로 발견하고 별도 Predictor가 예측합니다. 특성 입력은 0으로 두어 메타데이터 없이 정본 구조를 유지합니다.

In [ ]:
class ValueOfIndexLayer(tf.keras.layers.Layer):
    def __init__(self, index):
        super(ValueOfIndexLayer, self).__init__()
        self.index = index
        self.value = tf.Variable(initial_value=tf.zeros((1,)))

    def call(self, input):
        self.value = input[:, self.index]
        return self.value


## Store user and item latent values
Two Finders alternate user/item latent discovery, followed by a separate Predictor. Zero-valued feature slots retain the reference architecture without adding metadata.

두 Finder가 학생·문항 잠재값을 교대로 발견하고 별도 Predictor가 예측합니다. 특성 입력은 0으로 두어 메타데이터 없이 정본 구조를 유지합니다.

In [ ]:
class IndexedValueLayer(layers.Layer):
    def __init__(self, value_shape, trainable: bool, name: str,
                 latent_regularizer=None, latent_constraint=None,
                 latent_initializer="he_normal", **kwargs):
        super(IndexedValueLayer, self).__init__(name=name, **kwargs)
        vocab_size, latent_dims = value_shape
        self.input_dim = vocab_size
        self.output_dim = latent_dims
        self.kernel = self.add_weight(
            shape=(self.input_dim, self.output_dim),
            name=f"{self.name}.kernel", trainable=trainable,
            regularizer=latent_regularizer, constraint=latent_constraint,
            initializer=latent_initializer)

    def call(self, inputs):
        return tf.gather(self.kernel, tf.cast(inputs, tf.int32))

    def set_value(self, value):
        self.kernel.assign(value)

    def get_value(self):
        return self.kernel.numpy()


## Binary response loss with explicit label shape
Two Finders alternate user/item latent discovery, followed by a separate Predictor. Zero-valued feature slots retain the reference architecture without adding metadata.

두 Finder가 학생·문항 잠재값을 교대로 발견하고 별도 Predictor가 예측합니다. 특성 입력은 0으로 두어 메타데이터 없이 정본 구조를 유지합니다.

In [ ]:
class BatchBinaryCrossentropy(Loss):
    def call(self, y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        # label-shape fix carried in the published repo (GY, 2026-07-28)
        y_true = tf.reshape(tf.cast(y_true, y_pred.dtype), tf.shape(y_pred))
        bce = -(y_true * tf.math.log(y_pred) + (1 - y_true) * tf.math.log(1 - y_pred))
        return tf.reduce_mean(bce, axis=-1)


## The user and item Finders
Two Finders alternate user/item latent discovery, followed by a separate Predictor. Zero-valued feature slots retain the reference architecture without adding metadata.

두 Finder가 학생·문항 잠재값을 교대로 발견하고 별도 Predictor가 예측합니다. 특성 입력은 0으로 두어 메타데이터 없이 정본 구조를 유지합니다.

In [ ]:
def build_finder_model(user_latents_shape, item_features_shape, item_latents_shape,
                       user_latent_trainable=True, item_latent_trainable=True):
    inp = Input((2,), name="input")
    user_index = ValueOfIndexLayer(0)(inp)
    item_index = ValueOfIndexLayer(1)(inp)

    user_latents_layer = IndexedValueLayer(user_latents_shape, trainable=user_latent_trainable, name="user_latents")
    item_features_layer = IndexedValueLayer(item_features_shape, trainable=False, name="item_features")
    item_latents_layer = IndexedValueLayer(item_latents_shape, trainable=item_latent_trainable, name="item_latents")

    user_latent_x = user_latents_layer(user_index)
    item_feature_x = item_features_layer(item_index)
    item_latent_x = item_latents_layer(item_index)

    user_latent_x = Dense(96, activation="swish", use_bias=True, kernel_regularizer=l2(0.0001), name="user_latent")(user_latent_x)
    user_latent_x = BatchNormalization()(user_latent_x)
    user_latent_x = Dropout(0.2)(user_latent_x)

    item_feature_x = Dense(64, activation="swish", use_bias=True, kernel_regularizer=l2(0.0001), name="item_feature")(item_feature_x)
    item_feature_x = BatchNormalization()(item_feature_x)
    item_feature_x = Dropout(0.2)(item_feature_x)

    item_latent_x = Dense(96, activation="swish", use_bias=True, kernel_regularizer=l2(0.0001), name="item_latent")(item_latent_x)
    item_latent_x = BatchNormalization()(item_latent_x)
    item_latent_x = Dropout(0.2)(item_latent_x)

    x = concatenate([user_latent_x, item_feature_x, item_latent_x])
    x = BatchNormalization()(x)
    x = Dense(256, activation="swish", use_bias=True, kernel_regularizer=l2(0.0001))(x)
    x = BatchNormalization()(x); x = Dropout(0.3)(x)
    x = Dense(128, activation="swish", use_bias=True, kernel_regularizer=l2(0.0001))(x)
    x = BatchNormalization()(x); x = Dropout(0.3)(x)
    x = Dense(64, activation="swish", use_bias=True, kernel_regularizer=l2(0.0001))(x)
    x = BatchNormalization()(x); x = Dropout(0.2)(x)
    score_output = Dense(1, activation="sigmoid", kernel_regularizer=l2(0.0001))(x)

    model = Model(inp, score_output)
    model.user_latents_layer = user_latents_layer
    model.item_features_layer = item_features_layer
    model.item_latents_layer = item_latents_layer
    return model


## The separate Predictor
Two Finders alternate user/item latent discovery, followed by a separate Predictor. Zero-valued feature slots retain the reference architecture without adding metadata.

두 Finder가 학생·문항 잠재값을 교대로 발견하고 별도 Predictor가 예측합니다. 특성 입력은 0으로 두어 메타데이터 없이 정본 구조를 유지합니다.

In [ ]:
def build_predictor_model_optimized(user_feature_dim, item_feature_dim, latent_dim, name="Predictor"):
    user_latent_input = Input((latent_dim,), name="user_latent_input")
    item_feature_input = Input((item_feature_dim,), name="item_feature_input")
    item_latent_input = Input((latent_dim,), name="item_latent_input")
    user_feature_input = Input((user_feature_dim,), name="user_feature_input")

    user_latent_x = Dense(96, activation="swish", use_bias=True, kernel_regularizer=l2(0.0001), name="user_latent")(user_latent_input)
    user_latent_x = BatchNormalization()(user_latent_x); user_latent_x = Dropout(0.2)(user_latent_x)

    item_feature_x = Dense(64, activation="swish", use_bias=True, kernel_regularizer=l2(0.0001), name="item_feature")(item_feature_input)
    item_feature_x = BatchNormalization()(item_feature_x); item_feature_x = Dropout(0.2)(item_feature_x)

    item_latent_x = Dense(96, activation="swish", use_bias=True, kernel_regularizer=l2(0.0001), name="item_latent")(item_latent_input)
    item_latent_x = BatchNormalization()(item_latent_x); item_latent_x = Dropout(0.2)(item_latent_x)

    user_feature_x = Dense(4, activation="swish", use_bias=True, kernel_regularizer=l2(0.0001), name="user_feature")(user_feature_input)
    user_feature_x = BatchNormalization()(user_feature_x); user_feature_x = Dropout(0.1)(user_feature_x)

    x = concatenate([user_latent_x, item_feature_x, item_latent_x, user_feature_x])
    x = BatchNormalization()(x)
    x = Dense(256, activation="swish", use_bias=True, kernel_regularizer=l2(0.0001))(x)
    x = BatchNormalization()(x); x = Dropout(0.3)(x)
    x = Dense(128, activation="swish", use_bias=True, kernel_regularizer=l2(0.0001))(x)
    x = BatchNormalization()(x); x = Dropout(0.3)(x)
    x = Dense(64, activation="swish", use_bias=True, kernel_regularizer=l2(0.0001))(x)
    x = BatchNormalization()(x); x = Dropout(0.2)(x)
    score_output = Dense(1, activation="sigmoid", kernel_regularizer=l2(0.0001))(x)

    return Model(inputs=[user_latent_input, item_feature_input, item_latent_input, user_feature_input],
                 outputs=score_output, name=name)


## Read bounded batches from the response log
Index batches avoid expanding latent vectors for the entire dataset. The Predictor optimizes BCE only, matching the reference custom loop; the Finders retain their L2 terms.

전체 자료의 잠재값을 복제하지 않고 인덱스로 배치를 구성합니다. Predictor는 정본처럼 BCE만, Finder는 L2를 포함한 손실로 학습합니다.

In [ ]:
class ResponseBatches(tf.keras.utils.PyDataset):
    """Read only one batch of index pairs from the memory-mapped response log."""
    def __init__(self, data, rows, batch_size, shuffle, seed):
        super().__init__()
        self.data, self.rows = data, np.asarray(rows).copy()
        self.batch_size, self.shuffle = batch_size, shuffle
        self.rng = np.random.default_rng(seed)
        self.on_epoch_end()
    def __len__(self):
        return (len(self.rows) + self.batch_size - 1) // self.batch_size
    def __getitem__(self, index):
        rows = self.rows[index * self.batch_size:(index + 1) * self.batch_size]
        inputs = np.column_stack((self.data["person"][rows], self.data["item"][rows])).astype(np.float32)
        return inputs, self.data["correct"][rows].astype(np.float32).reshape(-1, 1)
    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.rows)


## Progress during long training steps
Index batches avoid expanding latent vectors for the entire dataset. The Predictor optimizes BCE only, matching the reference custom loop; the Finders retain their L2 terms.

전체 자료의 잠재값을 복제하지 않고 인덱스로 배치를 구성합니다. Predictor는 정본처럼 BCE만, Finder는 L2를 포함한 손실로 학습합니다.

In [ ]:
class TrainingProgress(tf.keras.callbacks.Callback):
    def __init__(self, label, interval=30):
        super().__init__()
        self.label, self.interval = label, interval
    def on_train_begin(self, logs=None):
        self.last_message = time.monotonic()
        print(f"Starting {self.label}", flush=True)
    def on_train_batch_end(self, batch, logs=None):
        if time.monotonic() - self.last_message >= self.interval:
            print(f"  {self.label}: batch {batch+1}, loss={logs['loss']:.5f}", flush=True)
            self.last_message = time.monotonic()
    def on_epoch_end(self, epoch, logs=None):
        if epoch == 0 or (epoch + 1) % 10 == 0 or "val_auc" in logs:
            values = {name:round(float(value), 5) for name,value in logs.items()}
            print(f"  {self.label}: epoch {epoch+1} {values}", flush=True)


## The exact Predictor training objective
Index batches avoid expanding latent vectors for the entire dataset. The Predictor optimizes BCE only, matching the reference custom loop; the Finders retain their L2 terms.

전체 자료의 잠재값을 복제하지 않고 인덱스로 배치를 구성합니다. Predictor는 정본처럼 BCE만, Finder는 L2를 포함한 손실로 학습합니다.

In [ ]:
class BCEPredictor(tf.keras.Model):
    """Use the reference Predictor objective: BCE, without declared L2 terms."""
    def __init__(self, inputs, outputs):
        super().__init__(inputs=inputs, outputs=outputs)
        self.loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.accuracy_tracker = tf.keras.metrics.BinaryAccuracy(name="accuracy")
        self.auc_tracker = tf.keras.metrics.AUC(name="auc")
    @property
    def metrics(self):
        return [self.loss_tracker, self.accuracy_tracker, self.auc_tracker]
    def train_step(self, batch):
        inputs, labels, weights = tf.keras.utils.unpack_x_y_sample_weight(batch)
        with tf.GradientTape() as tape:
            probabilities = self(inputs, training=True)
            loss_value = self.loss(labels, probabilities)
        gradients = tape.gradient(loss_value, self.trainable_weights)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_weights))
        return self.update_metrics(labels, probabilities, loss_value)
    def test_step(self, batch):
        inputs, labels, weights = tf.keras.utils.unpack_x_y_sample_weight(batch)
        probabilities = self(inputs, training=False)
        return self.update_metrics(labels, probabilities, self.loss(labels, probabilities))
    def update_metrics(self, labels, probabilities, loss_value):
        self.loss_tracker.update_state(loss_value)
        self.accuracy_tracker.update_state(labels, probabilities)
        self.auc_tracker.update_state(labels, probabilities)
        return {metric.name:metric.result() for metric in self.metrics}


## Validation-based stopping and learning rate
Index batches avoid expanding latent vectors for the entire dataset. The Predictor optimizes BCE only, matching the reference custom loop; the Finders retain their L2 terms.

전체 자료의 잠재값을 복제하지 않고 인덱스로 배치를 구성합니다. Predictor는 정본처럼 BCE만, Finder는 L2를 포함한 손실로 학습합니다.

In [ ]:
class ReferenceStopping(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.best, self.best_weights = 0.0, None
        self.stop_count = self.rate_count = 0
    def on_epoch_end(self, epoch, logs=None):
        if logs["val_auc"] > self.best:
            self.best, self.best_weights = logs["val_auc"], self.model.get_weights()
            self.stop_count = self.rate_count = 0
        else:
            self.stop_count += 1
            self.rate_count += 1
            if self.rate_count >= 3:
                reduced = float(self.model.optimizer.learning_rate.numpy()) * .5
                if reduced > 1e-6:
                    self.model.optimizer.learning_rate.assign(reduced)
                    self.rate_count = 0
        if self.stop_count >= 7:
            self.model.stop_training = True
    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)


## Connect discovered latents to the Predictor
Index batches avoid expanding latent vectors for the entire dataset. The Predictor optimizes BCE only, matching the reference custom loop; the Finders retain their L2 terms.

전체 자료의 잠재값을 복제하지 않고 인덱스로 배치를 구성합니다. Predictor는 정본처럼 BCE만, Finder는 L2를 포함한 손실로 학습합니다.

In [ ]:
def make_indexed_predictor(user_latents, item_latents, config):
    inputs = Input((2,), name="response_indices")
    user_index, item_index = ValueOfIndexLayer(0)(inputs), ValueOfIndexLayer(1)(inputs)
    user_layer = IndexedValueLayer(user_latents.shape, trainable=False, name="fixed_user_latents")
    item_layer = IndexedValueLayer(item_latents.shape, trainable=False, name="fixed_item_latents")
    zeros = layers.Lambda(lambda values: tf.zeros_like(values[:, :1]), name="zero_features")(inputs)
    core = build_predictor_model_optimized(1, 1, config["latent_size"])
    output = core([user_layer(user_index), zeros, item_layer(item_index), zeros])
    model = BCEPredictor(inputs, output)
    user_layer.set_value(user_latents)
    item_layer.set_value(item_latents)
    model.user_latents_layer, model.item_latents_layer = user_layer, item_layer
    return model


## Batches, objective and validation control
Index batches avoid expanding latent vectors for the entire dataset. The Predictor optimizes BCE only, matching the reference custom loop; the Finders retain their L2 terms.

전체 자료의 잠재값을 복제하지 않고 인덱스로 배치를 구성합니다. Predictor는 정본처럼 BCE만, Finder는 L2를 포함한 손실로 학습합니다.

In [ ]:


# Observational checkpointing; no training update or stopping rule is changed.
CHECKPOINT_DIRECTORY = None


## Batches, objective and validation control
Index batches avoid expanding latent vectors for the entire dataset. The Predictor optimizes BCE only, matching the reference custom loop; the Finders retain their L2 terms.

전체 자료의 잠재값을 복제하지 않고 인덱스로 배치를 구성합니다. Predictor는 정본처럼 BCE만, Finder는 L2를 포함한 손실로 학습합니다.

In [ ]:
def save_training_state(model, label, include_optimizer=True):
    if CHECKPOINT_DIRECTORY is None:
        return
    values = {f"model_{j}":v.numpy() for j,v in enumerate(model.variables)}
    metadata = {"model":[{"name":v.path,"shape":list(v.shape)} for v in model.variables]}
    if include_optimizer:
        values.update({f"optimizer_{j}":v.numpy() for j,v in enumerate(model.optimizer.variables)})
        metadata["optimizer"] = [{"name":v.path,"shape":list(v.shape)} for v in model.optimizer.variables]
    else:
        metadata["optimizer_note"] = "Predictor best weights restored by original callback; end optimizer is not paired with them and is not exported."
    path = CHECKPOINT_DIRECTORY / (label + "_state.npz")
    temporary = path.with_suffix(".pending.npz")
    np.savez(temporary, **values)
    temporary.replace(path)
    save_json(CHECKPOINT_DIRECTORY / (label + "_state.json"), metadata)


## Batches, objective and validation control
Index batches avoid expanding latent vectors for the entire dataset. The Predictor optimizes BCE only, matching the reference custom loop; the Finders retain their L2 terms.

전체 자료의 잠재값을 복제하지 않고 인덱스로 배치를 구성합니다. Predictor는 정본처럼 BCE만, Finder는 L2를 포함한 손실로 학습합니다.

In [ ]:

OriginalTrainingProgress = TrainingProgress


## Progress during long training steps
Index batches avoid expanding latent vectors for the entire dataset. The Predictor optimizes BCE only, matching the reference custom loop; the Finders retain their L2 terms.

전체 자료의 잠재값을 복제하지 않고 인덱스로 배치를 구성합니다. Predictor는 정본처럼 BCE만, Finder는 L2를 포함한 손실로 학습합니다.

In [ ]:
class TrainingProgress(OriginalTrainingProgress):
    def on_train_begin(self, logs=None):
        super().on_train_begin(logs)
        self.phase_started = time.monotonic()
    def on_train_end(self, logs=None):
        elapsed = time.monotonic() - self.phase_started
        if CHECKPOINT_DIRECTORY is None:
            return
        start = time.monotonic()
        if self.label.startswith("cycle"):
            save_training_state(self.model, "last_" + self.label.split()[-1])
        elif self.label == "Predictor":
            save_training_state(self.model, "best_Predictor", include_optimizer=False)
        path = CHECKPOINT_DIRECTORY / "phase_times.json"
        phases = json.loads(path.read_text()) if path.exists() else []
        phases.append({"phase":self.label,"fit_seconds":elapsed,"checkpoint_seconds":time.monotonic()-start})
        save_json(path, phases)


## Alternate user and item latent discovery
Each k starts from the same initial user latents with a fresh optimizer. Adaptation checks that every frozen weight and every other student’s latent remains unchanged.

각 k는 같은 초기 학생 잠재값과 새 옵티마이저에서 시작합니다. 적응 중 동결 가중치와 다른 학생의 잠재값이 변하지 않았는지 확인합니다.

In [ ]:
def fit_cdld(data, rows, valid_rows, description, config, seed):
    tf.keras.utils.set_random_seed(seed)
    n_people, n_items = description["people"], description["items"]
    size = config["latent_size"]
    initial_users = np.random.default_rng(seed).random((n_people, size)).astype(np.float32)
    initial_items = np.random.default_rng(seed + 1000).random((n_items, size)).astype(np.float32)
    zeros = np.zeros((n_items, 1), np.float32)
    user_finder = build_finder_model(initial_users.shape, zeros.shape, initial_items.shape, True, False)
    item_finder = build_finder_model(initial_users.shape, zeros.shape, initial_items.shape, False, True)
    for finder in [user_finder, item_finder]:
        finder.user_latents_layer.set_value(initial_users)
        finder.item_features_layer.set_value(zeros)
        finder.item_latents_layer.set_value(initial_items)
        finder.compile(optimizer=tf.keras.optimizers.Adam(config["learning_rate"]), loss=BatchBinaryCrossentropy(),
                       metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy"), tf.keras.metrics.AUC(name="auc")])
    training = ResponseBatches(data, rows, config["batch_size"], True, seed)
    validation = ResponseBatches(data, valid_rows, config["batch_size"], False, seed)
    schedule = tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=.5, patience=2, min_lr=1e-5)
    histories = []
    for cycle in range(config["finder_cycles"]):
        user_finder.item_latents_layer.set_value(item_finder.item_latents_layer.get_value())
        for label, finder in [("ULD", user_finder), ("ILD", item_finder)]:
            if label == "ILD":
                item_finder.user_latents_layer.set_value(user_finder.user_latents_layer.get_value())
            history = finder.fit(training, validation_data=validation, epochs=config["finder_subepochs"], verbose=0,
                                 callbacks=[schedule, TrainingProgress(f"cycle {cycle+1} {label}")])
            histories.append({"phase":label, "cycle":cycle+1, "values":history.history})
    final_users = user_finder.user_latents_layer.get_value()
    final_items = item_finder.item_latents_layer.get_value()
    user_finder.item_latents_layer.set_value(final_items)
    del item_finder
    gc.collect()
    predictor = make_indexed_predictor(final_users, final_items, config)
    predictor.compile(optimizer=tf.keras.optimizers.Adam(config["learning_rate"]), loss=BatchBinaryCrossentropy())
    history = predictor.fit(training, validation_data=validation, epochs=config["predictor_epochs"], verbose=0,
                            callbacks=[ReferenceStopping(), TrainingProgress("Predictor")])
    histories.append({"phase":"Predictor", "cycle":0, "values":history.history})
    return {"finder":user_finder, "predictor":predictor, "initial_users":initial_users,
            "final_users":final_users, "final_items":final_items, "histories":histories}


## Predict the held-out responses
Each k starts from the same initial user latents with a fresh optimizer. Adaptation checks that every frozen weight and every other student’s latent remains unchanged.

각 k는 같은 초기 학생 잠재값과 새 옵티마이저에서 시작합니다. 적응 중 동결 가중치와 다른 학생의 잠재값이 변하지 않았는지 확인합니다.

In [ ]:
def predict_cdld(fitted, data, query_rows, config):
    flat = query_rows.ravel()
    inputs = np.column_stack((data["person"][flat], data["item"][flat])).astype(np.float32)
    return fitted["predictor"].predict(inputs, batch_size=config["batch_size"], verbose=0).reshape(query_rows.shape)


## Adapt only a new student’s latent
Each k starts from the same initial user latents with a fresh optimizer. Adaptation checks that every frozen weight and every other student’s latent remains unchanged.

각 k는 같은 초기 학생 잠재값과 새 옵티마이저에서 시작합니다. 적응 중 동결 가중치와 다른 학생의 잠재값이 변하지 않았는지 확인합니다.

In [ ]:
def adapt_cdld(fitted, data, people, support_rows, config):
    finder = fitted["finder"]
    for layer in finder.layers:
        layer.trainable = False
    finder.user_latents_layer.trainable = True
    finder.compile(optimizer=tf.keras.optimizers.Adam(config["learning_rate"]), loss=BatchBinaryCrossentropy())
    if len(finder.trainable_weights) != 1:
        raise RuntimeError("Only the user-latent array may be adapted.")
    before = [(weight, weight.numpy().copy()) for layer in finder.layers
              if layer is not finder.user_latents_layer for weight in layer.weights]
    values = fitted["final_users"].copy()
    values[people] = fitted["initial_users"][people]
    finder.user_latents_layer.set_value(values)
    if support_rows.shape[1]:
        flat = support_rows.ravel()
        inputs = np.column_stack((data["person"][flat], data["item"][flat])).astype(np.float32)
        labels = data["correct"][flat].astype(np.float32).reshape(-1, 1)
        finder.fit(inputs, labels, batch_size=len(flat), epochs=config["adaptation_epochs"], verbose=0,
                   callbacks=[TrainingProgress(f"adaptation k={support_rows.shape[1]}")])
    for weight, old_value in before:
        np.testing.assert_array_equal(weight.numpy(), old_value)
    adapted = finder.user_latents_layer.get_value()
    other_people = np.ones(len(values), dtype=bool)
    other_people[people] = False
    np.testing.assert_array_equal(adapted[other_people], values[other_people])
    fitted["predictor"].user_latents_layer.set_value(adapted)
    return adapted[people]


# 5. Implementation checks

## Run the implementation checks before interpreting results
Check missing-response handling, label shape and the actual Predictor update. A failed check stops the notebook; it is not a low-scoring experiment.

미관측 응답 처리·라벨 형상·Predictor의 실제 갱신을 확인합니다. 검사 실패 시 노트북을 중단하고 결과를 해석하지 않습니다.

In [ ]:
def run_reproduction_gates():
    # Sparse likelihood: compare with a literal per-response calculation,
    # including missing entries and a person with no observed response.
    small = np.empty(4, dtype=RESPONSE_DTYPE)
    small["person"], small["item"], small["correct"] = [0, 0, 1, 1], [0, 2, 1, 2], [1, 0, 0, 1]
    small["timestamp"] = -1
    observed, correct = sparse_responses(small, np.arange(4), 3, 3)
    theta, prior = irt_grid(41)
    a, b = np.array([.7, 1.1, 1.5]), np.array([-.3, .2, 1.0])
    trials, successes, likelihood = irt_expectation(observed, correct, a, b, theta, prior, 2)
    reference_trials, reference_successes = np.zeros_like(trials), np.zeros_like(successes)
    for person in range(3):
        rows = np.flatnonzero(small["person"] == person)
        probability = expit(a[small["item"][rows], None] * (theta[None, :] - b[small["item"][rows], None]))
        labels = small["correct"][rows, None]
        log_probability = (labels * np.log(probability) + (1-labels) * np.log1p(-probability)).sum(axis=0) + np.log(prior)
        posterior = np.exp(log_probability - logsumexp(log_probability))
        for row in rows:
            reference_trials[small["item"][row]] += posterior
            reference_successes[small["item"][row]] += small["correct"][row] * posterior
    np.testing.assert_allclose(trials, reference_trials, atol=1e-12)
    np.testing.assert_allclose(successes, reference_successes, atol=1e-12)
    # BCE shape and an explicit BCE-only gradient comparison.
    loss = BatchBinaryCrossentropy()
    np.testing.assert_allclose(float(loss(tf.constant([0., 1.]), tf.constant([[.2], [.8]]))), -np.log(.8), rtol=1e-6)
    tf.keras.utils.set_random_seed(91)
    inputs = Input((2,))
    output = Dense(1, activation="sigmoid", kernel_regularizer=l2(.1))(inputs)
    model = BCEPredictor(inputs, output)
    model.compile(optimizer=tf.keras.optimizers.SGD(.01), loss=loss)
    values, labels = tf.constant([[1., 2.], [2., 1.]]), tf.constant([[0.], [1.]])
    original = [weight.numpy().copy() for weight in model.trainable_weights]
    with tf.GradientTape() as tape:
        expected_loss = loss(labels, model(values, training=True))
    gradients = tape.gradient(expected_loss, model.trainable_weights)
    model.train_step((values, labels))
    for old, gradient, weight in zip(original, gradients, model.trainable_weights):
        np.testing.assert_allclose(weight.numpy(), old - .01 * gradient.numpy(), atol=1e-7)
    print("PASS: missing-response likelihood, label shape, and BCE-only Predictor update.")
    tf.keras.backend.clear_session()
    gc.collect()
    return True

GATES_PASSED = run_reproduction_gates()


# 6. Execute the experiment

## What each saved number measures
AUC evaluates individual response ranking; score correlation evaluates the observed query-score fraction. Neither is a validation of latent true ability or an actual licensing decision.

AUC는 개별 응답 순위를, 점수 상관은 관측된 평가 문항 정답률 예측을 봅니다. 잠재 능력의 참값이나 실제 국시 합격 판정을 검증한 것이 아닙니다.

In [ ]:
def response_metrics(actual, probability):
    labels, predicted = actual.ravel(), np.clip(probability.ravel(), 1e-7, 1 - 1e-7)
    observed_score, predicted_score = actual.mean(axis=1), probability.mean(axis=1)
    correlation = float(np.corrcoef(observed_score, predicted_score)[0, 1]) if predicted_score.std() > 1e-12 and observed_score.std() > 1e-12 else np.nan
    return {"auc": float(roc_auc_score(labels, predicted)),
            "log_loss": float(log_loss(labels, predicted, labels=[0, 1])),
            "brier": float(brier_score_loss(labels, predicted)),
            "accuracy": float(((predicted >= .5) == labels).mean()),
            "score_correlation": correlation,
            "observed_correct_rate": float(labels.mean()),
            "score_mae": float(abs(observed_score - predicted_score).mean())}

def append_prediction(storage, metric_rows, model, k, probability, actual, protocol, seed):
    if not np.isfinite(probability).all() or (probability < 0).any() or (probability > 1).any():
        raise ValueError(f"Invalid predictions: {model}, k={k}")
    storage[f"probability|{model}|{k}"] = probability.astype(np.float32)
    metric_rows.append({"protocol":protocol, "seed":seed, "model":model, "k":k,
                        **response_metrics(actual, probability)})

def verify_completed(directory):
    marker = directory / "complete.json"
    if not marker.exists():
        return False
    checksums = json.loads(marker.read_text())
    for name, expected in checksums.items():
        if file_sha256(directory / name) != expected:
            raise RuntimeError(f"Saved result failed its checksum: {directory / name}")
    return True


## Fit and save one matched comparison
Fit every model on the same rows. Completed runs are verified and skipped; IRT can resume an iteration checkpoint, while an interrupted CDLD fit restarts its unfinished run.

모든 모형을 같은 행으로 학습합니다. 완료 반복은 검증 후 건너뛰며 IRT는 반복 계산 체크포인트, CDLD는 미완료 실행 단위에서 재개합니다.

In [ ]:
def run_one_experiment(data, description, split, config, seed, protocol, directory):
    directory.mkdir(exist_ok=True, parents=True)
    rows = split["plans"][protocol]["train"]
    valid_rows = split["plans"][protocol]["valid"]
    people, query = split["people"], split["query"]
    actual = data["correct"][query]
    results = {"person_ids":people, "query_items":data["item"][query], "actual":actual,
               "query_row_ids":query, "support_row_ids":split["support"],
               "query_timestamp":data["timestamp"][query]}
    metric_rows, model_diagnostics = [], {}
    save_npz(directory / "split.npz", train_rows=rows, validation_rows=valid_rows,
             evaluation_people=people, query_rows=query, support_rows=split["support"])
    save_json(directory / "split_audit.json", split["audit"])
    if protocol == "warm":
        training_people = data["person"][rows]
        support = [rows[np.searchsorted(training_people, person, side="left"):
                        np.searchsorted(training_people, person, side="right")] for person in people]
        counts = [-1]  # All available training responses; not a k-shot condition.
    else:
        support = split["support"]
        counts = config["support_counts"]
    # Reuse only independently verified, converged IRT fits on identical rows.
    reuse_root = WORK_DIR / "previous_results" / DATASET_NAME / RUN_MODE
    if reuse_root.exists():
        import shutil
        old_config = json.loads((reuse_root / "run_manifest.json").read_text())["config"]
        assert {k:v for k,v in old_config.items() if k != "finder_subepochs"} == {k:v for k,v in config.items() if k != "finder_subepochs"}
        old_manifest = json.loads((reuse_root / "run_manifest.json").read_text())
        assert old_manifest["response_sha256"] == description["response_sha256"]
        old = reuse_root / f"{protocol}_seed{seed}"
        hashes = json.loads((old / "complete.json").read_text())
        for name in ["split.npz", "model_diagnostics.json", "Rasch_checkpoint.npz", "2PL_checkpoint.npz"]:
            assert file_sha256(old / name) == hashes[name], name
        with np.load(old / "split.npz") as before, np.load(directory / "split.npz") as after:
            assert set(before.files) == set(after.files)
            for key in before.files:
                np.testing.assert_array_equal(before[key], after[key])
        diagnostics = json.loads((old / "model_diagnostics.json").read_text())
        for model in ["Rasch", "2PL"]:
            assert diagnostics[model]["converged"]
            with np.load(old / f"{model}_checkpoint.npz") as checkpoint:
                assert checkpoint["history"][-1,1] < config["irt_tolerance"]
            target = directory / f"{model}_checkpoint.npz"
            shutil.copy2(old / target.name, target)
        save_json(directory / "irt_reuse.json", {"source":str(old), "source_manifest_sha256":file_sha256(reuse_root / "run_manifest.json"), "verified_identical_split":True})
    rate = float(data["correct"][rows].mean())
    item_count = np.bincount(data["item"][rows], minlength=description["items"])
    item_correct = np.bincount(data["item"][rows], weights=data["correct"][rows], minlength=description["items"])
    item_rate = np.divide(item_correct, item_count, out=np.full_like(item_correct, rate), where=item_count > 0)
    for k in counts:
        append_prediction(results, metric_rows, "Constant", k, np.full(actual.shape, rate), actual, protocol, seed)
        append_prediction(results, metric_rows, "Item rate", k, item_rate[data["item"][query]], actual, protocol, seed)
    for model in ["Rasch", "2PL"]:
        fitted_irt = fit_irt(data, rows, description, config, model, directory / f"{model}_checkpoint.npz")
        model_diagnostics[model] = {key:fitted_irt[key] for key in ["converged", "iterations", "slope_boundary_items"]}
        pd.DataFrame({"item":np.arange(description["items"]), "a":fitted_irt["a"], "b":fitted_irt["b"],
                      "estimable":fitted_irt["active"]}).to_csv(directory / f"{model}_item_parameters.csv", index=False)
        pd.DataFrame(fitted_irt["history"], columns=["iteration", "max_change", "log_likelihood_before_update"]).to_csv(directory / f"{model}_history.csv", index=False)
        for k in counts:
            chosen = support if protocol == "warm" else support[:, :k]
            probability, theta = irt_predict(data, chosen, query, fitted_irt, config)
            results[f"theta|{model}|{k}"] = theta
            append_prediction(results, metric_rows, model, k, probability, actual, protocol, seed)
        del fitted_irt
        gc.collect()
    global CHECKPOINT_DIRECTORY
    CHECKPOINT_DIRECTORY = directory
    fitted_cdld = fit_cdld(data, rows, valid_rows, description, config, seed)
    save_json(directory / "CDLD_history.json", fitted_cdld["histories"])
    save_training_state(fitted_cdld["finder"], "Finder_before_adaptation", include_optimizer=True)
    np.save(directory / "CDLD_item_latents.npy", fitted_cdld["final_items"])
    # Latents are numeric analysis artifacts; no executable fitted model is exported.
    np.save(directory / "CDLD_user_latents.npy", fitted_cdld["final_users"])
    for k in counts:
        if protocol == "cold":
            results[f"adapted_latents|{k}"] = adapt_cdld(fitted_cdld, data, people, support[:, :k], config)
        probability = predict_cdld(fitted_cdld, data, query, config)
        append_prediction(results, metric_rows, "CDLD", k, probability, actual, protocol, seed)
    save_npz(directory / "predictions.npz", **results)
    pd.DataFrame(metric_rows).to_csv(directory / "metrics.csv", index=False)
    save_json(directory / "model_diagnostics.json", model_diagnostics)
    save_json(directory / "complete.json", {path.name:file_sha256(path) for path in directory.iterdir()
                                          if path.is_file() and path.name != "complete.json"})
    del fitted_cdld
    tf.keras.backend.clear_session()
    gc.collect()


## Resume completed runs and isolate failures
Fit every model on the same rows. Completed runs are verified and skipped; IRT can resume an iteration checkpoint, while an interrupted CDLD fit restarts its unfinished run.

모든 모형을 같은 행으로 학습합니다. 완료 반복은 검증 후 건너뛰며 IRT는 반복 계산 체크포인트, CDLD는 미완료 실행 단위에서 재개합니다.

In [ ]:
def run_experiments(data, description, config):
    notebook_document = json.loads(NOTEBOOK_PATH.read_text())
    actual_common_source = "\n".join("".join(cell["source"]) for cell in notebook_document["cells"]
                                     if cell["cell_type"] == "code" and cell.get("metadata", {}).get("shared"))
    actual_common_digest = hashlib.sha256(actual_common_source.encode()).hexdigest()
    manifest = {"dataset":DATASET_NAME, "mode":RUN_MODE, "config":config,
                "response_sha256":description["response_sha256"], "common_code_sha256":actual_common_digest}
    manifest_file = OUTPUT_DIR / "run_manifest.json"
    if manifest_file.exists() and json.loads(manifest_file.read_text()) != manifest:
        raise RuntimeError("Different data, code or settings already occupy this output directory. Choose a new WORK_DIR.")
    save_json(manifest_file, manifest)
    failures = []
    for seed in config["seeds"]:
        split = make_split(data, description, config, seed)
        for protocol in config["protocols"]:
            directory = OUTPUT_DIR / f"{protocol}_seed{seed}"
            if RESUME and verify_completed(directory):
                print("Verified completed run; skipping:", directory.name)
                continue
            if directory.exists() and not RESUME:
                raise FileExistsError("Existing result directory; enable RESUME or choose a new WORK_DIR.")
            pending = OUTPUT_DIR / (directory.name + ".pending")
            if RESUME and verify_completed(pending):
                pending.rename(directory)
                print("Recovered a completed pending run:", directory.name)
                continue
            try:
                started = time.monotonic()
                run_one_experiment(data, description, split, config, seed, protocol, pending)
                pending.rename(directory)
                print(f"Completed {directory.name} in {(time.monotonic()-started)/60:.1f} minutes", flush=True)
            except Exception as error:
                failures.append({"protocol":protocol, "seed":seed, "error":str(error)})
                save_json(OUTPUT_DIR / f"failure_{protocol}_{seed}.json", {"traceback":traceback.format_exc()})
                print(f"FAILED {protocol}, seed={seed}: {error}. Continuing remaining runs.", flush=True)
            finally:
                tf.keras.backend.clear_session()
                gc.collect()
        del split
        gc.collect()
    if failures:
        raise RuntimeError(f"{len(failures)} run(s) failed. Inspect failure files and rerun this cell to resume.")
    print("All requested runs are complete and checksummed.")


## Start the configured runs
Fit every model on the same rows. Completed runs are verified and skipped; IRT can resume an iteration checkpoint, while an interrupted CDLD fit restarts its unfinished run.

모든 모형을 같은 행으로 학습합니다. 완료 반복은 검증 후 건너뛰며 IRT는 반복 계산 체크포인트, CDLD는 미완료 실행 단위에서 재개합니다.

In [ ]:

if RUN_EXPERIMENT:
    assert GATES_PASSED
    run_experiments(responses, data_description, CONFIG)
else:
    print("Training skipped. Continue to the saved-results analysis and figures.")


# 7. Results from saved predictions

## Saved-results analysis — no retraining
Resample people jointly across repeats to preserve identity dependence. Conditional intervals and the observed-repeat sensitivity analysis are reported separately.

같은 사람을 반복 간 함께 재추출합니다. 고정된 적합에 조건부인 구간과 관찰된 반복 변동을 추가한 분석을 구분합니다.

In [ ]:
# This cell reads saved predictions only. It does not need a fitted model or raw data.
def analyze_saved_results():
    import hashlib
    from scipy.special import ndtr
    def checksum(path):
        value = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(8*1024*1024), b""):
                value.update(block)
        return value.hexdigest()
    def interval(values):
        return np.nanpercentile(values, [5, 2.5, 97.5])
    def weighted_correlation(weights, x, y):
        count = weights.sum(axis=1)
        sx, sy = weights @ x, weights @ y
        vx = weights @ (x*x) - sx*sx/count
        vy = weights @ (y*y) - sy*sy/count
        covariance = weights @ (x*y) - sx*sy/count
        return covariance / np.sqrt(np.maximum(vx, 1e-20)*np.maximum(vy, 1e-20))
    def auc_influence(actual, probability):
        labels, values = actual.ravel().astype(bool), probability.ravel()
        positive, negative = np.sort(values[labels]), np.sort(values[~labels])
        auc = roc_auc_score(labels, values)
        influence = np.empty(len(values))
        influence[labels] = ((np.searchsorted(negative, values[labels], "left") + np.searchsorted(negative, values[labels], "right"))/(2*len(negative)) - auc)/len(positive)
        influence[~labels] = (1-(np.searchsorted(positive, values[~labels], "left") + np.searchsorted(positive, values[~labels], "right"))/(2*len(positive)) - auc)/len(negative)
        return auc, influence.reshape(actual.shape).sum(axis=1)
    all_metrics, comparisons, calibration_rows = [], [], []
    for protocol in CONFIG["protocols"]:
        saved, diagnostics = [], []
        for seed in CONFIG["seeds"]:
            directory = OUTPUT_DIR / f"{protocol}_seed{seed}"
            marker = directory / "complete.json"
            if not marker.exists():
                raise RuntimeError(f"Missing completed run: {directory.name}. Partial results are not a full experiment.")
            for name, expected in json.loads(marker.read_text()).items():
                if checksum(directory/name) != expected:
                    raise RuntimeError(f"Checksum failure: {directory/name}")
            all_metrics.append(pd.read_csv(directory / "metrics.csv"))
            saved.append(np.load(directory / "predictions.npz"))
            diagnostics.append(json.loads((directory / "model_diagnostics.json").read_text()))
        universe = np.unique(np.concatenate([result["person_ids"] for result in saved]))
        rng = np.random.default_rng(CONFIG["bootstrap_seed"])
        weights = rng.multinomial(len(universe), np.full(len(universe), 1/len(universe)), size=CONFIG["bootstrap_draws"]).astype(float)
        n_repeats = len(saved)
        repeat_counts = rng.multinomial(n_repeats, np.full(n_repeats, 1/n_repeats), size=CONFIG["bootstrap_draws"])
        counts = [-1] if protocol == "warm" else CONFIG["support_counts"]
        for k in counts:
            for comparator in ["Rasch", "2PL"]:
                score_draws, auc_draws, score_points, auc_points = [], [], [], []
                for result in saved:
                    actual = result["actual"]
                    cdld = result[f"probability|CDLD|{k}"].astype(float)
                    baseline = result[f"probability|{comparator}|{k}"].astype(float)
                    person_weights = weights[:, np.searchsorted(universe, result["person_ids"])]
                    target, ours, theirs = actual.mean(axis=1), cdld.mean(axis=1), baseline.mean(axis=1)
                    score_points.append(float(np.corrcoef(ours, target)[0,1]-np.corrcoef(theirs,target)[0,1]) if ours.std()>1e-12 and theirs.std()>1e-12 else np.nan)
                    score_draws.append(weighted_correlation(person_weights, ours, target)-weighted_correlation(person_weights, theirs, target))
                    our_auc, our_influence = auc_influence(actual, cdld)
                    their_auc, their_influence = auc_influence(actual, baseline)
                    auc_points.append(our_auc-their_auc)
                    auc_draws.append(our_auc-their_auc + (person_weights-1) @ (our_influence-their_influence))
                converged = all(item[comparator]["converged"] for item in diagnostics)
                for metric, points, draws in [("score_correlation",score_points,score_draws), ("auc",auc_points,auc_draws)]:
                    if not np.isfinite(points).all():
                        continue
                    samples = np.asarray(draws).T
                    conditional = interval(samples.mean(axis=1))
                    with_repeats = interval((samples * repeat_counts).sum(axis=1)/n_repeats)
                    comparisons.append({"dataset":DATASET_NAME, "protocol":protocol, "k":k, "comparator":comparator,
                        "metric":metric, "difference":float(np.mean(points)), "lower_one_sided95":conditional[0],
                        "lower95":conditional[1], "upper95":conditional[2],
                        "repeat_lower_one_sided95":with_repeats[0], "repeat_lower95":with_repeats[1], "repeat_upper95":with_repeats[2],
                        "baseline_converged":converged,
                        "conditional_noninferior":bool(conditional[0] > -CONFIG["correlation_margin"]) if metric=="score_correlation" and k>0 and converged else None})
            if k == counts[-1]:
                for seed, result in zip(CONFIG["seeds"], saved):
                    for model in ["Constant", "Item rate", "Rasch", "2PL", "CDLD"]:
                        probability, actual = result[f"probability|{model}|{k}"].ravel(), result["actual"].ravel()
                        bins = np.minimum((probability*10).astype(int), 9)
                        for group in range(10):
                            mask = bins == group
                            if mask.any():
                                calibration_rows.append({"protocol":protocol,"seed":seed,"model":model,"bin":group,
                                    "count":int(mask.sum()),"predicted":float(probability[mask].mean()),"observed":float(actual[mask].mean())})
        for result in saved:
            result.close()
    metrics_frame = pd.concat(all_metrics, ignore_index=True)
    metrics_frame.insert(0, "dataset", DATASET_NAME)
    metrics_frame.to_csv(OUTPUT_DIR / "metrics_all.csv", index=False)
    summary = metrics_frame.groupby(["dataset","protocol","model","k"], as_index=False).agg(
        auc=("auc","mean"), auc_sd=("auc","std"), score_correlation=("score_correlation","mean"),
        score_correlation_sd=("score_correlation","std"), log_loss=("log_loss","mean"), brier=("brier","mean"),
        accuracy=("accuracy","mean"), score_mae=("score_mae","mean"))
    summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
    comparison_frame = pd.DataFrame(comparisons)
    comparison_frame.to_csv(OUTPUT_DIR / "paired_comparisons.csv", index=False)
    pd.DataFrame(calibration_rows).to_csv(OUTPUT_DIR / "calibration.csv", index=False)
    display(summary.round(4))
    display(comparison_frame.loc[(comparison_frame["k"]==max(CONFIG["support_counts"])) & (comparison_frame["comparator"]=="2PL")].round(5))
    print("Inference is conditional on saved fits. AUC intervals use a first-order person-cluster approximation.")
    print("Repeat resampling uses the observed runs; it is not a full bootstrap that retrains every model.")
    print("Score correlation concerns the observed 40-item query score, not a latent true ability or a licensing decision.")

analyze_saved_results()


# 8. Figures for the manuscript

## Paper figures — no model or raw data needed
Read the exported CSVs to rebuild every figure after a kernel restart. PNG and vector PDF files are saved with the numerical source tables.

커널 재시작 후 저장 CSV만으로 그림을 다시 만듭니다. 수치 원본과 함께 300dpi PNG와 벡터 PDF를 저장합니다.

In [ ]:
# Figure-only cell: Setup + paths/settings + this cell are sufficient after restart.
summary_for_plot = pd.read_csv(OUTPUT_DIR / "summary.csv")
comparisons_for_plot = pd.read_csv(OUTPUT_DIR / "paired_comparisons.csv")
calibration_for_plot = pd.read_csv(OUTPUT_DIR / "calibration.csv")
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(exist_ok=True)
plot_colors = {"CDLD":"#0072B2", "2PL":"#D55E00", "Rasch":"#009E73", "Item rate":"#777777", "Constant":"#AAAAAA"}

def save_paper_figure(figure, name):
    if RUN_MODE != "full":
        figure.text(.5, .01, "PIPELINE CHECK — NOT PAPER RESULTS", ha="center", color="#AA3333", fontsize=9)
    figure.tight_layout(rect=(0,.04,1,1))
    for extension in ["png", "pdf"]:
        figure.savefig(FIGURE_DIR / f"{name}.{extension}", dpi=300, bbox_inches="tight")
    display(figure)
    plt.close(figure)

cold_rows = summary_for_plot.loc[summary_for_plot["protocol"]=="cold"]
if len(cold_rows):
    print("[Figure 1] New-student response prediction and observed query-score correlation. Bars show repeat SD, not confidence intervals.")
    figure, axes = plt.subplots(1, 2, figsize=(9, 3.7))
    for model in ["Constant", "Item rate", "Rasch", "2PL", "CDLD"]:
        selected = cold_rows.loc[cold_rows["model"]==model].sort_values("k")
        for axis, metric, ylabel in zip(axes, ["auc","score_correlation"], ["Response AUC", "Observed query-score correlation"]):
            axis.errorbar(selected["k"], selected[metric], yerr=selected[metric+"_sd"].fillna(0),
                          color=plot_colors[model], label=model, marker="o", markersize=3, linewidth=1.4, capsize=2)
            axis.set_xlabel("Number of adaptation responses")
            axis.set_ylabel(ylabel)
            axis.set_xticks(CONFIG["support_counts"])
    axes[0].legend(frameon=False, fontsize=8)
    save_paper_figure(figure, "fig1_cold_performance")
    print("[Figure 2] CDLD minus 2PL. Intervals preserve person identity across repeats and are conditional on saved fits.")
    figure, axes = plt.subplots(1, 2, figsize=(9, 3.7))
    for axis, metric, ylabel in zip(axes, ["auc","score_correlation"], ["AUC difference", "Query-score correlation difference"]):
        selected = comparisons_for_plot.loc[(comparisons_for_plot["protocol"]=="cold") &
                    (comparisons_for_plot["comparator"]=="2PL") & (comparisons_for_plot["metric"]==metric)].sort_values("k")
        axis.axhline(0, color="#444444", linewidth=.8)
        axis.plot(selected["k"], selected["difference"], color=plot_colors["CDLD"], marker="o", markersize=4)
        axis.fill_between(selected["k"], selected["lower95"], selected["upper95"], color=plot_colors["CDLD"], alpha=.18)
        axis.plot(selected["k"], selected["repeat_lower95"], color=plot_colors["CDLD"], linestyle=":", linewidth=1)
        axis.plot(selected["k"], selected["repeat_upper95"], color=plot_colors["CDLD"], linestyle=":", linewidth=1)
        if metric == "score_correlation":
            axis.axhline(-CONFIG["correlation_margin"], color="#777777", linestyle="--", linewidth=.9)
        axis.set_xlabel("Number of adaptation responses")
        axis.set_ylabel(ylabel)
    save_paper_figure(figure, "fig2_paired_differences")
print("[Figure 3] Reliability of predicted probabilities. Perfect calibration lies on the diagonal; points are descriptive aggregates.")
figure, axes = plt.subplots(1, len(CONFIG["protocols"]), figsize=(4.5*len(CONFIG["protocols"]), 3.8), squeeze=False)
for axis, protocol in zip(axes.ravel(), CONFIG["protocols"]):
    axis.plot([0,1], [0,1], color="#999999", linestyle="--", linewidth=1)
    for model in ["Rasch", "2PL", "CDLD"]:
        selected = calibration_for_plot.loc[(calibration_for_plot["protocol"]==protocol) & (calibration_for_plot["model"]==model)].copy()
        selected["predicted_sum"] = selected["predicted"] * selected["count"]
        selected["observed_sum"] = selected["observed"] * selected["count"]
        grouped = selected.groupby("bin")[["predicted_sum", "observed_sum", "count"]].sum()
        axis.plot(grouped["predicted_sum"]/grouped["count"], grouped["observed_sum"]/grouped["count"],
                  color=plot_colors[model], marker="o", markersize=4, label=model)
    axis.set_xlabel(f"Predicted probability ({protocol})")
    axis.set_ylabel("Observed correct fraction")
    axis.set_xlim(0,1)
    axis.set_ylim(0,1)
axes[0,0].legend(frameon=False)
save_paper_figure(figure, "fig3_probability_calibration")
print("Saved PNG (300 dpi), vector PDF, and the CSV data used for these figures:", FIGURE_DIR)


# 9. Reuse and verification

## Files for verification and additional analyses
Inspect the inventory before transferring results. Response-level labels, IDs and latent arrays are research data, not automatically public release material.

전송 전에 파일 목록을 확인합니다. 응답·ID·잠재값은 연구자료이며 자동으로 공개 배포 가능한 자료를 뜻하지 않습니다.

In [ ]:
# Preserve decoding metadata next to results so transferred outputs remain interpretable.
for mapping_name in ["people.csv", "items.csv", "complete.json"]:
    mapping_source = CACHE_DIR / mapping_name
    if mapping_source.exists():
        output_name = "data_description.json" if mapping_name == "complete.json" else mapping_name
        (OUTPUT_DIR / output_name).write_bytes(mapping_source.read_bytes())
import importlib.metadata
if not (OUTPUT_DIR / "environment.json").exists():
    environment_record = {
        "python": sys.version,
        "packages": {name: importlib.metadata.version(name) for name in
                     ["tensorflow", "keras", "numpy", "scipy", "scikit-learn", "pandas", "matplotlib"]},
        "recorded_utc": datetime.now(timezone.utc).isoformat(),
    }
    (OUTPUT_DIR / "environment.json").write_text(json.dumps(environment_record, indent=2))

# Do not include response-level artifacts in a public upload without checking data terms.
artifact_rows = []
for artifact in sorted(OUTPUT_DIR.rglob("*")):
    if artifact.is_file():
        artifact_rows.append({"file":str(artifact.relative_to(OUTPUT_DIR)), "size_MB":round(artifact.stat().st_size/1024**2, 3)})
artifact_inventory = pd.DataFrame(artifact_rows)
artifact_inventory.to_csv(OUTPUT_DIR / "artifact_inventory.csv", index=False)
display(artifact_inventory)
print("Repeat predictions include IDs, query labels, support/query row indices, probabilities, IRT estimates and adapted CDLD latents.")
print("Split files, convergence histories and numeric latent arrays support additional analyses without retraining.")


## Completion / 완료
`DONE` means the cells above finished. Paper results require full mode, every requested run, and converged comparator fits.
위 셀들이 끝났음을 표시합니다. 본실험 여부·모든 반복의 완료·IRT 수렴을 별도로 확인해야 합니다.

In [ ]:
print("DONE")
